In [ ]:
# 1. Install required packages
!pip install orb-models
!pip install openbabel

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 416.2/416.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 464.2/464.2 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.3/164.3 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 7.4 MB/s eta 0:00:00
  Created wheel for dm-tree: filename=dm_tree-0.1.8-cp313-cp313-linux_x86_64.whl size=135921 sha256=3d8fb4a7879a09a20fe71212f47afaf93668d299f9b920820633b7018b04a960
  Stored in directory: /root/.cache/pip/wheels/67/a2/15/a6aa1b4f084c1b7092f68044144c2b7ffab0a894bde001b6e8
Successfully built dm-tree
  Attempting uninstall: dm-tree
    Found existing installation: dm-tree 0.1.10
    Uninstalling dm-tr

In [ ]:
import os
import warnings
import numpy as np
from ase.io import read, write
from ase import Atoms, units
from ase.md.langevin import Langevin
from ase.constraints import FixAtoms

warnings.filterwarnings("ignore")

from orb_models.forcefield import pretrained
from orb_models.forcefield.inference.calculator import ORBCalculator

# =================================================================
# CONFIGURATION VARIABLES
# =================================================================
SURFACE_XYZ = "surface.xyz"     # Path to base substrate
IMPLANT_SPECIES = "Ar"          # Implant species
ENERGY_EV = 1000                # Ion kinetic energy (eV)
TARGET_FLUENCE = 5.0e14         # Target total fluence in ions/cm^2

DT_FS = 0.5                     # Time step: 0.5 fs
STAGE_1_PS = 2.0                # Stage 1 duration: 2 ps
STAGE_2_PS = 3.0                # Stage 2 duration: 3 ps (Total = 5 ps)

FRICTION_STAGE_1 = 0.01         # Cascade friction (0 to 2 ps)
FRICTION_STAGE_2 = 0.1          # Thermal quench friction (2 to 5 ps)
TEMP_K = 300.0                  # Heat sink temperature (K)
FIXED_CUTOFF_Z = 3.0            # Bottom fixed slab thickness (Å)
LOG_INTERVAL = 1                # Output step interval for trajectory and thermo logs
# =================================================================

def prune_above_z(atoms, cutoff_z, fixed_indices, log_file_handle=None):
    """Removes atoms strictly above cutoff_z and realigns fixed atom indices."""
    keep_indices = [atom.index for atom in atoms if atom.z <= cutoff_z]
    n_removed = len(atoms) - len(keep_indices)

    if n_removed > 0:
        msg = f"  [Cleanup] Pruned {n_removed} atom(s) above z = {cutoff_z:.2f} Å (sputtered/backscattered).\n"
        print(msg.strip())
        if log_file_handle:
            log_file_handle.write(f"# {msg}")
            log_file_handle.flush()

        atoms = atoms[keep_indices]
        # Map existing fixed atom indices to their new positions in the sliced Atoms object
        new_fixed = [i for i, old_idx in enumerate(keep_indices) if old_idx in fixed_indices]
        atoms.set_constraint(FixAtoms(indices=new_fixed))
        return atoms, new_fixed

    return atoms, fixed_indices

def run_cyclic_implantation():
    print(f"Loading surface from {SURFACE_XYZ}...")
    surface = read(SURFACE_XYZ)

    # Lateral cell setup
    pos = surface.positions
    min_x, max_x = np.min(pos[:, 0]), np.max(pos[:, 0])
    min_y, max_y = np.min(pos[:, 1]), np.max(pos[:, 1])
    min_z = np.min(pos[:, 2])

    if np.all(surface.cell == 0):
        xy_buffer = 2.5
        surface.set_cell([
            max_x - min_x + xy_buffer,
            max_y - min_y + xy_buffer,
            np.max(pos[:, 2]) - min_z
        ])
        surface.center(axis=(0, 1))

    # Shift bottom of substrate to z = 0
    surface.positions[:, 2] -= np.min(surface.positions[:, 2])
    surface.wrap()

    # Save original surface max z before any ion impacts
    initial_surface_max_z = float(np.max(surface.positions[:, 2]))
    print(f"Initial substrate max Z height: {initial_surface_max_z:.3f} Å")

    # Expand Z vacuum buffer
    cell = surface.get_cell()
    cell[2, 2] = initial_surface_max_z + 1000.0
    surface.set_cell(cell)
    surface.set_pbc([True, True, True])

    # Calculate cross-sectional area and fluence step size
    area_angstrom2 = np.linalg.norm(np.cross(cell[0], cell[1]))
    area_cm2 = area_angstrom2 * 1e-16
    fluence_per_ion = 1.0 / area_cm2
    total_cycles = int(np.ceil(TARGET_FLUENCE / fluence_per_ion))

    print(f"Cross-sectional area: {area_angstrom2:.2f} Å² ({area_cm2:.2e} cm²)")
    print(f"Fluence per ion: {fluence_per_ion:.2e} ions/cm²")
    print(f"Target fluence: {TARGET_FLUENCE:.2e} ions/cm² -> Executing {total_cycles} cycle(s)")

    # Fixed bottom anchor layer (0 to 3.0 Å)
    fixed_indices = [atom.index for atom in surface if atom.z < FIXED_CUTOFF_Z]
    surface.set_constraint(FixAtoms(indices=fixed_indices))

    # Initialize ORB ML Calculator
    device = "cuda"
    print("Loading ORB ML force field...")
    orbff, atoms_adapter = pretrained.orb_v3_direct_20_omat(device=device, precision="float32-high")
    calc = ORBCalculator(orbff, atoms_adapter=atoms_adapter, device=device)

    # Master simulation trajectory
    xyz_file = 'implantation.extxyz'
    if os.path.exists(xyz_file):
        os.remove(xyz_file)

    system = surface.copy()
    current_fluence = 0.0

    # Steps calculation
    dt = DT_FS * units.fs
    steps_stage_1 = int((STAGE_1_PS * 1000.0) / DT_FS)  # 4000 steps
    steps_stage_2 = int((STAGE_2_PS * 1000.0) / DT_FS)  # 6000 steps

    for cycle in range(1, total_cycles + 1):
        current_fluence += fluence_per_ion
        print(f"\n=== Cycle {cycle}/{total_cycles} | Fluence: {current_fluence:.2e} ions/cm² ===")

        current_max_z = np.max(system.positions[:, 2])

        # Randomize lateral insertion point
        edge_buf = 3.0
        len_x, len_y = np.linalg.norm(cell[0]), np.linalg.norm(cell[1])
        fx = np.random.uniform(edge_buf / len_x, 1.0 - (edge_buf / len_x))
        fy = np.random.uniform(edge_buf / len_y, 1.0 - (edge_buf / len_y))

        # Position ion 8 Å above the highest existing atom
        ion_pos = fx * cell[0] + fy * cell[1]
        ion_pos[2] = current_max_z + 8.0
        implant_atom = Atoms(f"{IMPLANT_SPECIES}1", positions=[ion_pos])

        # Compute downward velocity for the chosen kinetic energy
        mass = implant_atom[0].mass
        v_mag = np.sqrt(2 * ENERGY_EV / mass) * (units.fs / units.Angstrom)
        v_ion = np.zeros((1, 3))
        v_ion[0, 2] = -v_mag

        # Preserve existing thermal velocities and add projectile
        old_v = system.get_velocities()
        if old_v is None:
            old_v = np.zeros((len(system), 3))

        system += implant_atom
        system.set_velocities(np.vstack((old_v, v_ion)))
        system.set_constraint(FixAtoms(indices=fixed_indices))
        system.calc = calc
        system.info["charge"] = 0
        system.info["spin"] = 1

        def save_snapshot():
            snap = system.copy()
            snap.wrap()
            write(xyz_file, snap, format='extxyz', append=True)

        # Write cycle log file tracking thermodynamics, temperature, and energies
        cycle_log = f"implantation_cycle_{cycle}.log"
        with open(cycle_log, "w") as log_file:
            log_file.write(f"# Cycle {cycle}/{total_cycles} - Fluence: {current_fluence:.4e} ions/cm^2\n")
            log_file.write(f"# Stage 1: 0-2 ps (gamma={FRICTION_STAGE_1}), Stage 2: 2-5 ps (gamma={FRICTION_STAGE_2})\n")
            log_file.flush()

            # ---------------------------------------------------------
            # STAGE 1: 0 to 2 ps (Friction = 0.01)
            # ---------------------------------------------------------
            log_file.write("# --- START STAGE 1 (0 to 2 ps) ---\n")
            dyn1 = Langevin(
                system,
                dt,
                temperature_K=TEMP_K,
                friction=FRICTION_STAGE_1,
                fixcm=False,
                logfile=log_file,
                loginterval=LOG_INTERVAL
            )
            dyn1.attach(save_snapshot, interval=LOG_INTERVAL)
            dyn1.run(steps_stage_1)

            # Cleanup above initial surface max Z at 2 ps
            system, fixed_indices = prune_above_z(
                system,
                initial_surface_max_z,
                fixed_indices,
                log_file_handle=log_file
            )
            system.calc = calc

            # ---------------------------------------------------------
            # STAGE 2: 2 to 5 ps (Friction = 0.1)
            # ---------------------------------------------------------
            log_file.write("# --- START STAGE 2 (2 to 5 ps) ---\n")
            dyn2 = Langevin(
                system,
                dt,
                temperature_K=TEMP_K,
                friction=FRICTION_STAGE_2,
                fixcm=False,
                logfile=log_file,
                loginterval=LOG_INTERVAL
            )
            dyn2.attach(save_snapshot, interval=LOG_INTERVAL)
            dyn2.run(steps_stage_2)

            # Cleanup above initial surface max Z at 5 ps
            system, fixed_indices = prune_above_z(
                system,
                initial_surface_max_z,
                fixed_indices,
                log_file_handle=log_file
            )
            system.calc = calc

        print(f"Cycle {cycle} complete. Log saved to '{cycle_log}'.")

    print(f"\nTarget fluence reached: {current_fluence:.2e} ions/cm² across {total_cycles} cycles.")
    print(f"Full trajectory written to '{xyz_file}'.")

if __name__ == "__main__":
    run_cyclic_implantation()

Loading surface from surface.xyz...
Initial substrate max Z height: 79.225 Å
Cross-sectional area: 836.32 Å² (8.36e-14 cm²)
Fluence per ion: 1.20e+13 ions/cm²
Target fluence: 5.00e+14 ions/cm² -> Executing 42 cycle(s)
Loading ORB ML force field...

=== Cycle 1/42 | Fluence: 1.20e+13 ions/cm² ===
